In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfin_clientes
WHERE cl_base = 'mayo 2026'
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

In [3]:
filename='RetiroDeGestion_BlackList.csv'

filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
df_list.count()


DNI    430673
dtype: int64

In [4]:
print(df_list.columns)
print(df_dni.columns)

Index(['DNI'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [5]:
df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [6]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)
df_list['retiro']='Retirar RCC'
df_list.count()


dni_cliente    1886
retiro         1886
dtype: int64

In [7]:
df_list['retiro']='Retirar BlackList'


In [9]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=1000,
    validar_sin_grabar=False
)


Total registros a procesar: 1886
Lote 0 - 1000 actualizado | filas afectadas: 1000
Lote 1000 - 1886 actualizado | filas afectadas: 886
Proceso terminado. Total filas afectadas: 1886
